<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Projekt-Templates & MVP
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** M26a zeigte die Integration Pipeline für das Meeting- & Research-Briefing-System selbst. Dieses Notebook macht den Transfer auf **eigene** Projekte: wiederverwendbare Architektur-Templates (Research, Analyse, Support) und die Frage, wann ein Multi-Agent-System als **MVP** gilt.

> **Voraussetzung:** M26a — Integration Pipeline (Security Gate, Worker-Agenten, Quality Judge).

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M26b-Projekt-Templates"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M26_Integration_Pipeline",
    "tags": ["m26", "pipeline"],
    "metadata": {"notebook": "M26", "version": "1.0"}
}


# 1 | Übersicht
---

**M26a** hat eine konkrete Pipeline gebaut. Dieses Notebook abstrahiert davon: drei **Architektur-Templates** (A: Research/Recherche, B: Datenanalyse, C: Support/FAQ) als Ausgangspunkt für eigene Multi-Agent-Systeme, gefolgt von einer **MVP-Checkliste**, die klärt, wann ein System lauffähig, getestet und dokumentiert genug für einen ersten produktiven Einsatz ist.

Dieses Modul setzt **M26a_Integration_Pipeline** voraus: Die End-to-End-Pipeline ist bekannt; hier wird daraus ein Template-Baukasten für eigene Projekte.


# 2 | Transfer: Projekt-Templates
---


Dieser Abschnitt ist ein Transfer-Baukasten. Die eigentliche M26-Pipeline bleibt Security → Research → Writing → Judge; die Templates zeigen nur, wie dasselbe Muster auf andere Projekttypen übertragen wird.

Drei bewährte Templates decken die häufigsten Multi-Agent-Anwendungsfälle ab.
Jedes Template ist ein vollständiger Supervisor-Graph – direkt einsetzbar.

<p><font color='black' size="5">Template A – Content Pipeline</font></p>

**Einsatz:** Artikel, Berichte, Zusammenfassungen, Blogposts  
**Agents:** Recherche → Schreiben  
**Kernidee:** Fakten zuerst sammeln, dann strukturiert verfassen.


In [168]:
#@markdown   <p><font size="4" color='green'>  Template A – Content Pipeline</font> </br></p>

diag_a = '''
%%{init: {'theme':'light'}}%%
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nContent-Koordinator"]

    SUP -->|"recherche"| R["🔍 Recherche-Agent\nFakten sammeln"]
    SUP -->|"schreiben"| W["✍️ Schreib-Agent\nText erstellen"]
    SUP -->|"FINISH"| END([END])

    R -->|"AIMessage\n(name=Recherche)"| SUP
    W -->|"AIMessage\n(name=Schreiben)"| SUP

    R --- RT1["📚 wikipedia_suche"]
    W --- WT1["📋 gliederung_erstellen"]
    W --- WT2["🔢 wort_zaehlen"]

    style SUP fill:#FF9800,color:#000
    style R   fill:#2196F3,color:#fff
    style W   fill:#4CAF50,color:#fff
    style RT1 fill:#1565C0,color:#fff
    style WT1 fill:#2E7D32,color:#fff
    style WT2 fill:#2E7D32,color:#fff
'''
mermaid(diag_a, width=700)

In [169]:
#@markdown   <p><font size="4" color='green'>  Template A – Implementierung</font> </br></p>

# ── Template A: Content Pipeline ──────────────────────────────────────────
@tool
def wikipedia_suche(thema: str) -> str:
    '''Sucht Informationen zu einem Thema (simuliert Wikipedia).'''
    wissen = {
        "langchain": "LangChain: Framework für LLM-Anwendungen. Chains, Agents, Tools, LCEL.",
        "langgraph": "LangGraph: Zustandsbasierte Agenten-Graphen mit StateGraph und Checkpointing.",
        "rag":       "RAG: Retrieval-Augmented Generation kombiniert Vektorsuche mit LLM-Generierung.",
        "langsmith": "LangSmith: Tracing, Evaluation und Monitoring für LLM-Pipelines.",
    }
    key = thema.lower().split()[0].rstrip("?.,:")
    return wissen.get(key, f'Allgemeine Informationen zu "{thema}" gefunden.')

@tool
def gliederung_erstellen(thema: str, stichpunkte: str) -> str:
    '''Erstellt eine strukturierte Gliederung für einen Artikel.'''
    return (
        f'Gliederung zu "{thema}":\n'
        f'1. Einleitung\n2. Kernkonzept\n3. Anwendung\n4. Fazit\n\n'
        f'Basis: {stichpunkte[:120]}'
    )

@tool
def wort_zaehlen(text: str) -> str:
    '''Zählt Wörter in einem Text.'''
    n = len(text.split())
    return f'{n} Wörter | {len(text)} Zeichen'

class ContentState(TypedDict):
    thema: str
    recherche: str
    gliederung: str
    artikel: str

def recherche_node_a(state: ContentState) -> dict:
    return {"recherche": wikipedia_suche.invoke({"thema": state["thema"]})}

def schreiben_node_a(state: ContentState) -> dict:
    gliederung = gliederung_erstellen.invoke({"thema": state["thema"], "stichpunkte": state["recherche"]})
    artikel = (
        f"Kurzartikel zu {state['thema']}\n"
        f"Recherche: {state['recherche']}\n"
        f"{gliederung}\n"
        "Fazit: Das Template trennt Faktenrecherche und Texterstellung klar voneinander."
    )
    return {"gliederung": gliederung, "artikel": artikel + "\n" + wort_zaehlen.invoke({"text": artikel})}

content_builder_a = StateGraph(ContentState)
content_builder_a.add_node("recherche", recherche_node_a)
content_builder_a.add_node("schreiben", schreiben_node_a)
content_builder_a.add_edge(START, "recherche")
content_builder_a.add_edge("recherche", "schreiben")
content_builder_a.add_edge("schreiben", END)
content_supervisor = content_builder_a.compile()

print("✅ Template A: Content Pipeline bereit (StateGraph: Recherche → Schreiben)")


✅ Template A: Content Pipeline bereit (StateGraph: Recherche → Schreiben)


In [170]:
#@markdown   <p><font size="4" color='green'>  Template A – Demo</font> </br></p>

result_a = content_supervisor.invoke(
    {"thema": "LangChain", "recherche": "", "gliederung": "", "artikel": ""},
    config={
        "recursion_limit": 10,
        "run_name": "M26-TemplateA-Demo",
        "tags": ["m26", "template-a", "content-pipeline"],
    },
)
mprint("### 📝 Template A – Content Pipeline\n")
mprint(result_a["artikel"])


### 📝 Template A – Content Pipeline


Kurzartikel zu LangChain
Recherche: LangChain: Framework für LLM-Anwendungen. Chains, Agents, Tools, LCEL.
Gliederung zu "LangChain":
1. Einleitung
2. Kernkonzept
3. Anwendung
4. Fazit

Basis: LangChain: Framework für LLM-Anwendungen. Chains, Agents, Tools, LCEL.
Fazit: Das Template trennt Faktenrecherche und Texterstellung klar voneinander.
41 Wörter | 343 Zeichen


<p><font color='black' size="5">Template B – Analyse Pipeline</font></p>

**Einsatz:** Datenanalyse, Code-Reviews, Auswertungen  
**Agents:** Erfassen → Analysieren  
**Kernidee:** Rohdaten aufnehmen, dann auswerten und interpretieren.


In [171]:
#@markdown   <p><font size="4" color='green'>  Template B – Analyse Pipeline</font> </br></p>

diag_b = '''
%%{init: {'theme':'light'}}%%
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nAnalyse-Koordinator"]

    SUP -->|"erfassen"| E["📥 Erfassungs-Agent\nDaten aufnehmen"]
    SUP -->|"analysieren"| A["🔬 Analyse-Agent\nAuswerten"]
    SUP -->|"FINISH"| END([END])

    E -->|"AIMessage\n(name=Erfassen)"| SUP
    A -->|"AIMessage\n(name=Analyse)"| SUP

    E --- ET1["📊 daten_laden"]
    A --- AT1["▶️ python_ausfuehren"]
    A --- AT2["🔍 muster_suchen"]

    style SUP fill:#FF9800,color:#000
    style E   fill:#9C27B0,color:#fff
    style A   fill:#F44336,color:#fff
    style ET1 fill:#6A1B9A,color:#fff
    style AT1 fill:#B71C1C,color:#fff
    style AT2 fill:#B71C1C,color:#fff
'''
mermaid(diag_b, width=700)

In [172]:
#@markdown   <p><font size="4" color='green'>  Template B – Implementierung</font> </br></p>

import builtins
import re

# ── Template B: Analyse Pipeline ──────────────────────────────────────────
@tool
def daten_laden(quelle: str) -> str:
    '''Lädt Rohdaten aus einer Quelle (simuliert).'''
    daten = {
        "sales":   "Q1: 120k€, Q2: 145k€, Q3: 98k€, Q4: 167k€",
        "traffic": "Jan: 12.400 Besucher, Feb: 9.800, Mar: 15.600",
        "fehler":  "TypeError: 42x, KeyError: 18x, TimeoutError: 7x",
    }
    key = quelle.lower().split()[0].rstrip("?.,:")
    return daten.get(key, f'Daten aus "{quelle}": [100, 200, 150, 300, 250]')

@tool
def python_ausfuehren(ausdruck: str) -> str:
    '''Wertet einen einfachen Python-Ausdruck sicher aus.'''
    try:
        erlaubt = {k: getattr(builtins, k, None)
                   for k in ('sum', 'max', 'min', 'len', 'round', 'abs', 'sorted')}
        erlaubt = {k: v for k, v in erlaubt.items() if v is not None}
        return f'Ergebnis: {eval(ausdruck, {"__builtins__": erlaubt})}'
    except Exception as e:
        return f'Auswertungsfehler: {e}'

@tool
def muster_suchen(text: str, stichwort: str = "max") -> str:
    '''Sucht nach einem Stichwort im Text und gibt passende Zeilen zurück.'''
    treffer = [z for z in text.split(",") if stichwort.lower() in z.lower()]
    return f'Treffer für "{stichwort}": {treffer[0].strip() if treffer else "nicht gefunden"}'

template_b_tool_names = [daten_laden.name, python_ausfuehren.name, muster_suchen.name]
assert all(name.replace('_', '').replace('-', '').isalnum() for name in template_b_tool_names), template_b_tool_names

class AnalyseState(TypedDict):
    anfrage: str
    rohdaten: str
    analyse: str
    bericht: str

def _template_b_quelle(anfrage: str) -> str:
    text = anfrage.lower()
    if "traffic" in text or "besucher" in text:
        return "traffic"
    if "fehler" in text or "error" in text:
        return "fehler"
    return "sales"

def erfassen_node_b(state: AnalyseState) -> dict:
    quelle = _template_b_quelle(state["anfrage"])
    rohdaten = daten_laden.invoke({"quelle": quelle})
    return {"rohdaten": rohdaten}

def analysieren_node_b(state: AnalyseState) -> dict:
    werte = [(q, int(v)) for q, v in re.findall(r"(Q[1-4]):\s*(\d+)k", state["rohdaten"])]
    if werte:
        bestes_quartal, bester_wert = max(werte, key=lambda item: item[1])
        berechnung = python_ausfuehren.invoke({"ausdruck": f"max({[v for _, v in werte]})"})
        muster = muster_suchen.invoke({"text": ", ".join(f"{q}: {v}k" for q, v in werte), "stichwort": bestes_quartal})
        analyse = f"Bestes Quartal: {bestes_quartal} mit {bester_wert}k€. {berechnung}. {muster}."
    else:
        analyse = "Keine Quartalswerte erkannt; Daten wurden erfasst, aber nicht quantitativ ausgewertet."
    return {"analyse": analyse}

def bericht_node_b(state: AnalyseState) -> dict:
    bericht = (
        "Analyse-Bericht\n"
        f"Anfrage: {state['anfrage']}\n"
        f"Rohdaten: {state['rohdaten']}\n"
        f"Ergebnis: {state['analyse']}"
    )
    return {"bericht": bericht}

analyse_builder_b = StateGraph(AnalyseState)
analyse_builder_b.add_node("erfassen", erfassen_node_b)
analyse_builder_b.add_node("analysieren", analysieren_node_b)
analyse_builder_b.add_node("bericht", bericht_node_b)
analyse_builder_b.add_edge(START, "erfassen")
analyse_builder_b.add_edge("erfassen", "analysieren")
analyse_builder_b.add_edge("analysieren", "bericht")
analyse_builder_b.add_edge("bericht", END)
analyse_supervisor = analyse_builder_b.compile()

print("✅ Template B: Analyse Pipeline bereit (StateGraph: Erfassen → Analysieren → Bericht)")


✅ Template B: Analyse Pipeline bereit (StateGraph: Erfassen → Analysieren → Bericht)


In [173]:
#@markdown   <p><font size="4" color='green'>  Template B – Demo</font> </br></p>

result_b = analyse_supervisor.invoke(
    {
        "anfrage": "Analysiere die Sales-Daten und finde das beste Quartal.",
        "rohdaten": "",
        "analyse": "",
        "bericht": "",
    },
    config={
        "recursion_limit": 10,
        "run_name": "M26-TemplateB-Demo",
        "tags": ["m26", "template-b", "analyse-pipeline"],
    },
)
mprint("### 📊 Template B – Analyse Pipeline\n")
mprint(result_b["bericht"])


### 📊 Template B – Analyse Pipeline


Analyse-Bericht
Anfrage: Analysiere die Sales-Daten und finde das beste Quartal.
Rohdaten: Q1: 120k€, Q2: 145k€, Q3: 98k€, Q4: 167k€
Ergebnis: Bestes Quartal: Q4 mit 167k€. Ergebnis: 167. Treffer für "Q4": Q4: 167k.


<p><font color='black' size="5">Template C – Support Pipeline</font></p>

**Einsatz:** Ticket-Bearbeitung, FAQ-Systeme, Kundenservice  
**Agents:** Klassifizieren → Lösen  
**Kernidee:** Anfrage kategorisieren, dann zielgerichtet beantworten.

In [174]:
#@markdown   <p><font size="4" color='green'>  Template C – Support Pipeline</font> </br></p>

diag_c = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nSupport-Koordinator"]

    SUP -->|"klassifizieren"| K["🏷️ Klassifizier-Agent\nKategorie bestimmen"]
    SUP -->|"loesen"| L["🔧 Lösungs-Agent\nAntwort generieren"]
    SUP -->|"FINISH"| END([END])

    K -->|"AIMessage\n(name=Klassifizierung)"| SUP
    L -->|"AIMessage\n(name=Loesung)"| SUP

    K --- KT1["🗂️ ticket_kategorisieren"]
    L --- LT1["📖 faq_suchen"]
    L --- LT2["✅ antwort_validieren"]

    style SUP fill:#FF9800,color:#000
    style K   fill:#00BCD4,color:#000
    style L   fill:#8BC34A,color:#000
    style KT1 fill:#006064,color:#fff
    style LT1 fill:#33691E,color:#fff
    style LT2 fill:#33691E,color:#fff
'''
mermaid(diag_c, width=700)

In [175]:
#@markdown   <p><font size="4" color='green'>  Template C – Implementierung</font> </br></p>

# ── Template C: Support Pipeline ──────────────────────────────────────────
@tool
def ticket_kategorisieren(ticket: str) -> str:
    '''Kategorisiert ein Support-Ticket nach Typ und Priorität.'''
    text = ticket.lower()
    if any(w in text for w in ["error", "fehler", "absturz", "funktioniert nicht", "bug"]):
        return "Kategorie: Technisch | Priorität: Hoch"
    if any(w in text for w in ["rechnung", "zahlung", "preis", "kosten", "abonnement"]):
        return "Kategorie: Billing | Priorität: Mittel"
    return "Kategorie: Allgemein | Priorität: Niedrig"

@tool
def faq_suchen(frage: str) -> str:
    '''Sucht eine passende FAQ-Antwort.'''
    faq = {
        "passwort":    "Passwort zurücksetzen: Einstellungen → Sicherheit → Passwort ändern.",
        "rechnung":    "Rechnungen: Konto → Abrechnungen → Rechnungshistorie.",
        "installieren": "Installation: Setup-Datei laden, als Administrator ausführen.",
        "fehler":      "Fehler: App neu starten, Cache leeren, ggf. Support kontaktieren.",
    }
    for key, antwort in faq.items():
        if key in frage.lower():
            return antwort
    return f'Kein FAQ-Treffer für "{frage[:50]}". Weiterleitung an Support-Team.'

@tool
def antwort_validieren(antwort: str) -> str:
    '''Prüft, ob eine Support-Antwort vollständig und hilfreich ist.'''
    if len(antwort) < 20:
        return f'⚠️ Antwort zu kurz — bitte ergänzen.'
    return f'✅ Validiert ({len(antwort)} Zeichen): {antwort[:80]}...'

class SupportState(TypedDict):
    anfrage: str
    kategorie: str
    faq_antwort: str
    antwort: str

def klassifizieren_node_c(state: SupportState) -> dict:
    return {"kategorie": ticket_kategorisieren.invoke({"ticket": state["anfrage"]})}

def loesen_node_c(state: SupportState) -> dict:
    faq_antwort = faq_suchen.invoke({"frage": state["anfrage"]})
    validierung = antwort_validieren.invoke({"antwort": faq_antwort})
    antwort = f"{state['kategorie']}\nEmpfehlung: {faq_antwort}\n{validierung}"
    return {"faq_antwort": faq_antwort, "antwort": antwort}

support_builder_c = StateGraph(SupportState)
support_builder_c.add_node("klassifizieren", klassifizieren_node_c)
support_builder_c.add_node("loesen", loesen_node_c)
support_builder_c.add_edge(START, "klassifizieren")
support_builder_c.add_edge("klassifizieren", "loesen")
support_builder_c.add_edge("loesen", END)
support_supervisor = support_builder_c.compile()

print("✅ Template C: Support Pipeline bereit (StateGraph: Klassifizieren → Lösen)")


✅ Template C: Support Pipeline bereit (StateGraph: Klassifizieren → Lösen)


In [176]:
#@markdown   <p><font size="4" color='green'>  Template C – Demo</font> </br></p>

result_c = support_supervisor.invoke(
    {
        "anfrage": "Ich bekomme einen Fehler beim Starten der App. Was kann ich tun?",
        "kategorie": "",
        "faq_antwort": "",
        "antwort": "",
    },
    config={
        "recursion_limit": 10,
        "run_name": "M26-TemplateC-Demo",
        "tags": ["m26", "template-c", "support-pipeline"],
    },
)
mprint("### 🎫 Template C – Support Pipeline\n")
mprint(result_c["antwort"])


### 🎫 Template C – Support Pipeline


Kategorie: Technisch | Priorität: Hoch
Empfehlung: Fehler: App neu starten, Cache leeren, ggf. Support kontaktieren.
✅ Validiert (65 Zeichen): Fehler: App neu starten, Cache leeren, ggf. Support kontaktieren....

**Template-Vergleich:**

| | Template A | Template B | Template C |
|---|---|---|---|
| **Name** | Content Pipeline | Analyse Pipeline | Support Pipeline |
| **Agents** | Recherche, Schreiben | Erfassen, Analysieren | Klassifizieren, Lösen |
| **Input** | Thema / Frage | Daten / Code | Anfrage / Ticket |
| **Output** | Strukturierter Text | Auswertung / Bericht | Kategorisierte Antwort |
| **Typisches Tool** | wikipedia_suche | python_ausfuehren | faq_suchen |
| **Komplexität** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ |

<p><font color='black' size="5">
Template wählen
</font></p>

Die Wahl des richtigen Templates hängt vom **Output-Typ** ab:

- **Text/Inhalt erzeugen** → Template A (Content)
- **Daten/Code auswerten** → Template B (Analyse)
- **Anfragen bearbeiten** → Template C (Support)

Nutze den Entscheidungsbaum als erste Orientierung:

In [177]:
#@markdown   <p><font size="4" color='green'>  Template-Entscheidungsbaum</font> </br></p>

diag_e = '''
%%{init: {'theme':'light'}}%%
flowchart TD
    START(["❓ Mein Anwendungsfall"]) --> Q1{"Geht es um\nText-Erstellung?"}

    Q1 -->|"JA"| Q2{"Brauche ich\nFakten-Recherche?"}
    Q1 -->|"NEIN"| Q3{"Geht es um\nDatenauswertung?"}

    Q2 -->|"JA"| A["✅ Template A\nContent Pipeline"]
    Q2 -->|"NEIN"| A2["✅ Template A\n(nur Schreib-Agent)"]

    Q3 -->|"JA"| B["✅ Template B\nAnalyse Pipeline"]
    Q3 -->|"NEIN"| Q4{"Bearbeite ich\nAnfragen?"}

    Q4 -->|"JA"| C["✅ Template C\nSupport Pipeline"]
    Q4 -->|"NEIN"| CUSTOM["⚙️ Eigenes Template\nSiehe Aufgaben 5"]

    style A      fill:#4CAF50,color:#fff
    style A2     fill:#4CAF50,color:#fff
    style B      fill:#F44336,color:#fff
    style C      fill:#00BCD4,color:#000
    style CUSTOM fill:#FF9800,color:#000
    style Q1     fill:#37474F,color:#fff
    style Q2     fill:#37474F,color:#fff
    style Q3     fill:#37474F,color:#fff
    style Q4     fill:#37474F,color:#fff
'''
mermaid(diag_e, width=780)

# 3 | MVP-Definition
---

Ein **MVP (Minimum Viable Product)** für ein Multi-Agent-System ist lauffähig,
getestet und klar dokumentiert – aber noch nicht production-ready.



<p><font color='black' size="5">MVP-Checkliste</font></p>

| Kriterium | Beschreibung | Status Integration_Pipeline |
|-----------|-------------|---------------|
| ✅ **Läuft durch** | Graph endet ohne Fehler | Getestet |
| ✅ **Iterations-Schutz** | `max_iter` + `recursion_limit` | Implementiert |
| ✅ **Strukturiertes Routing** | `with_structured_output` | Implementiert |
| ✅ **Prompts extern** | `load_prompt()` statt Inline | Implementiert |
| ✅ **LangSmith-Tracing** | `run_name` + `tags` | Konfiguriert |
| ⚠️ **Fehlerbehandlung** | Try/Except in Worker-Nodes | Fehlt noch |
| ⚠️ **Checkpointing** | SQLite-Persistenz | Fehlt noch |
| ⚠️ **Tests** | Unit-Tests für Tools | Fehlt noch |
| ❌ **UI** | Gradio / Streamlit | Nicht in MVP |



<p><font color='black' size="5">Was kommt nach dem MVP?</font></p>

- ***OpenAI Agent Builder:*** Alternativer Ansatz ohne LangGraph-Boilerplate
- ***Agentic RAG:*** RAG-Agenten mit adaptivem Retrieval und Multi-Hop-Reasoning
- ***Gradio UI für Agenten:*** Web-Interface direkt für den eigenen Agenten
- ***Production Deployment:*** Fehlerbehandlung, Tests, Docker, Monitoring

<p><font color='darkblue' size="4">💡 <b>Tipp</b></font></p>

Starte immer mit dem MVP. Es ist besser, ein funktionierendes System mit 2 Agents zu haben als ein perfektes System das nie fertig wird.

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> Generative KI darf und soll im Kurs auch als Lernunterstützung genutzt werden — z. B. Gemini in Google Colab, um Fehlermeldungen zu verstehen, Teilschritte zu klären oder Code-Varianten zu prüfen.

**Grundlagen**

Baue für einen eigenen Use Case (Research Report, Evaluation Dashboard oder Review Gate) ein vollständiges Multi-Agent-System mit Template A, B oder C. Supervisor, Worker und Quality Judge müssen vorhanden sein.

**✅ Erledigt wenn:** Das System läuft für den gewählten Use Case end-to-end durch und gibt einen strukturierten Output aus — Architektur entspricht dem Template.

In [180]:
# Aufbau: Eigenes Research-System mit Template A, B oder C

USE_CASE = "research_report"  # Alternativen: evaluation_dashboard, review_gate


def waehle_template(use_case: str) -> dict:
    mapping = {
        "research_report": {
            "template": "Template A",
            "zweck": "belegte Research-Antwort aus mehreren Quellen",
            "pflicht_nodes": ["security_gate", "research", "writing", "quality_judge", "fact_checker"],
        },
        "evaluation_dashboard": {
            "template": "Template B",
            "zweck": "mehrere Fragen mit Qualitätsmetriken vergleichen",
            "pflicht_nodes": ["dataset", "runner", "metric_collector", "regression_gate"],
        },
        "review_gate": {
            "template": "Template C",
            "zweck": "kritische Ausgaben vor Veröffentlichung freigeben",
            "pflicht_nodes": ["risk_check", "draft", "review", "release_gate"],
        },
    }
    return mapping[use_case]

mein_system_plan = waehle_template(USE_CASE)
print(mein_system_plan)

{'template': 'Template A', 'zweck': 'belegte Research-Antwort aus mehreren Quellen', 'pflicht_nodes': ['security_gate', 'research', 'writing', 'quality_judge', 'fact_checker']}


In [181]:
# ✅ Selbstcheck Aufbau
assert mein_system_plan["template"] in {"Template A", "Template B", "Template C"}, "Unbekanntes Template."
assert len(mein_system_plan["pflicht_nodes"]) >= 4, "Der Systemplan braucht mindestens vier Pflicht-Nodes."
assert "zweck" in mein_system_plan and mein_system_plan["zweck"], "Zweckbeschreibung fehlt."
print("✅ Aufbau-Selfcheck bestanden.")

✅ Aufbau-Selfcheck bestanden.


**Vertiefung**

Wähle eine der zwei Optionen:

**Option A — Gradio UI mit Streaming:** Baue eine Gradio-App um die Pipeline, die den Fortschritt der Agenten in Echtzeit im UI anzeigt.

**Option B — Fehlertoleranz:** Ersetze `make_worker_node()` durch eine fehlertolerante Version mit Retry-Logik und sauberem Fallback bei Timeout oder API-Fehler.

**✅ Erledigt wenn — Option A:** Die Gradio-App zeigt den aktuellen Agent-Schritt in Echtzeit — kein leerer Output bis zum Ende.
**✅ Erledigt wenn — Option B:** Der Worker gibt bei einem erzwungenen Fehler eine strukturierte Fehlermeldung zurück — kein unbehandelter Traceback.

In [182]:
# Vertiefung: Fehlertolerante Worker-Node

def make_robust_worker_node(name: str, worker_fn, retries: int = 2):
    """Erzeugt eine Worker-Node mit Retry-Logik und sauberem Fallback."""
    def node(state: dict) -> dict:
        errors = []
        for attempt in range(1, retries + 1):
            try:
                value = worker_fn(state)
                return {**state, name: value, f"{name}_status": "ok", f"{name}_attempts": attempt}
            except Exception as exc:
                errors.append(f"attempt {attempt}: {type(exc).__name__}: {exc}")
        return {**state, name: "", f"{name}_status": "fallback", f"{name}_errors": errors}
    return node


def unstable_worker(state: dict) -> str:
    if state.get("force_error"):
        raise TimeoutError("simulierter Timeout")
    return "Worker-Ergebnis: Quellen geprüft und Antwort vorbereitet."

robust_worker = make_robust_worker_node("worker_output", unstable_worker, retries=2)
robust_ok = robust_worker({"force_error": False})
robust_fail = robust_worker({"force_error": True})
print("ok:", robust_ok)
print("fail:", robust_fail)

ok: {'force_error': False, 'worker_output': 'Worker-Ergebnis: Quellen geprüft und Antwort vorbereitet.', 'worker_output_status': 'ok', 'worker_output_attempts': 1}
fail: {'force_error': True, 'worker_output': '', 'worker_output_status': 'fallback', 'worker_output_errors': ['attempt 1: TimeoutError: simulierter Timeout', 'attempt 2: TimeoutError: simulierter Timeout']}


In [183]:
# ✅ Selbstcheck Vertiefung
assert robust_ok["worker_output_status"] == "ok", "Erfolgsfall muss ok sein."
assert robust_fail["worker_output_status"] == "fallback", "Fehlerfall muss in den Fallback gehen."
assert len(robust_fail["worker_output_errors"]) == 2, "Beide Retry-Versuche müssen protokolliert sein."
print("✅ Vertiefung-Selfcheck bestanden.")

✅ Vertiefung-Selfcheck bestanden.


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [Checkliste Automatisierung](https://editor.p5js.org/ralf.bendig.rb/full/ckiLlKrql)
- [KI-Prozessoptimierung](https://editor.p5js.org/ralf.bendig.rb/full/xGKXCTR7T)
- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
- [Minimum Viable Agent Stack](https://ralf-42.github.io/Agenten/08-deployment-betrieb/minimum-viable-agent-stack.html)
- [Code Standards](https://ralf-42.github.io/Agenten/10-ressourcen/standards.html)
